<a href="https://colab.research.google.com/github/TianyiPeng/vllm-tail-optimized-caching/blob/wenxin-edits/Tail_Optimized_VLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/TianyiPeng/vllm-tail-optimized-caching.git
%cd vllm-tail-optimized-caching

Cloning into 'vllm-tail-optimized-caching'...
remote: Enumerating objects: 73765, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 73765 (delta 0), reused 0 (delta 0), pack-reused 73762 (from 2)
Receiving objects: 100% (73765/73765), 52.04 MiB | 31.35 MiB/s, done.
Resolving deltas: 100% (57735/57735), done.
/content/vllm-tail-optimized-caching


In [ ]:
!VLLM_USE_PRECOMPILED=1 pip install -v --editable . #it may take 10+ minute

Streaming output truncated to the last 5000 lines.
    Skipping link: none of the wheel's tags (cp36-none-macosx_10_7_x86_64) are compatible (run pip debug --verbose to show compatible tags): https://files.pythonhosted.org/packages/d0/47/068dc7020fd8cf89cca74a16690e274fb55c08684bcaece11348d98264f0/torch-1.0.0-cp36-none-macosx_10_7_x86_64.whl (from https://pypi.org/simple/torch/)
    Skipping link: none of the wheel's tags (cp37-cp37m-manylinux1_x86_64) are compatible (run pip debug --verbose to show compatible tags): https://files.pythonhosted.org/packages/f5/3b/0b8de6e654c2983898564226792c6f09d9bcaba97b7b29c40e4ed4ae43ed/torch-1.0.0-cp37-cp37m-manylinux1_x86_64.whl (from https://pypi.org/simple/torch/)
    Skipping link: none of the wheel's tags (cp37-none-macosx_10_7_x86_64) are compatible (run pip debug --verbose to show compatible tags): https://files.pythonhosted.org/packages/32/44/3a9e5b9e7d582625880fbdee9f2162357c7f0c1776b29f1eb35bcc5357e8/torch-1.0.0-cp37-none-macosx_10_7_x86_6

In [ ]:
import sys
sys.path

['/content',
 '/env/python',
 '/usr/lib/python311.zip',
 '/usr/lib/python3.11',
 '/usr/lib/python3.11/lib-dynload',
 '',
 '/usr/local/lib/python3.11/dist-packages',
 '__editable__.vllm-0.1.dev6662+gdeaa125.precompiled.finder.__path_hook__',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.11/dist-packages/IPython/extensions',
 '/usr/local/lib/python3.11/dist-packages/setuptools/_vendor',
 '/root/.ipython']

## Please Restart the Session After the Installation and Before Running the Following Script

In [ ]:
!ls
import vllm
print(vllm.__file__)

sample_data  vllm-tail-optimized-caching
INFO 05-24 05:20:09 [__init__.py:248] Automatically detected platform cuda.
/content/vllm-tail-optimized-caching/vllm/__init__.py


In [ ]:
vllm

<module 'vllm' from '/content/vllm-tail-optimized-caching/vllm/__init__.py'>

In [ ]:
import time

from vllm import LLM, SamplingParams

# ruff: noqa: E501
# A prompt containing a large markdown table. The table is randomly generated by GPT-4.
LONG_PROMPT = "You are a helpful assistant in recognizes the content of tables in markdown format. Here is a table as follows.\n# Table\n" + """
| ID  | Name          | Age | Occupation    | Country       | Email                  | Phone Number   | Address                       |
|-----|---------------|-----|---------------|---------------|------------------------|----------------|------------------------------|
| 1   | John Doe      | 29  | Engineer      | USA           | john.doe@example.com   | 555-1234       | 123 Elm St, Springfield, IL  |
| 2   | Jane Smith    | 34  | Doctor        | Canada        | jane.smith@example.com | 555-5678       | 456 Oak St, Toronto, ON      |
| 3   | Alice Johnson | 27  | Teacher       | UK            | alice.j@example.com    | 555-8765       | 789 Pine St, London, UK      |
| 4   | Bob Brown     | 45  | Artist        | Australia     | bob.b@example.com      | 555-4321       | 321 Maple St, Sydney, NSW    |
| 5   | Carol White   | 31  | Scientist     | New Zealand   | carol.w@example.com    | 555-6789       | 654 Birch St, Wellington, NZ |
| 6   | Dave Green    | 28  | Lawyer        | Ireland       | dave.g@example.com     | 555-3456       | 987 Cedar St, Dublin, IE     |
| 7   | Emma Black    | 40  | Musician      | USA           | emma.b@example.com     | 555-1111       | 246 Ash St, New York, NY     |
| 8   | Frank Blue    | 37  | Chef          | Canada        | frank.b@example.com    | 555-2222       | 135 Spruce St, Vancouver, BC |
| 9   | Grace Yellow  | 50  | Engineer      | UK            | grace.y@example.com    | 555-3333       | 864 Fir St, Manchester, UK   |
"""


def get_generation_time(llm, sampling_params, prompts):
    # time the generation
    start_time = time.time()
    output = llm.generate(prompts, sampling_params=sampling_params)
    end_time = time.time()
    # print the output and generation time
    print("-" * 30)
    print(f"Output: {output[0].outputs[0].text}")
    print(f"Generation time: {end_time - start_time} seconds.")
    print("-" * 30)


In [ ]:
llm = LLM(
    model="facebook/opt-125m",
    enable_prefix_caching=True,
    caching_low_priority_last_num_tokens=32  # You can adjust this value based on your needs
)

INFO 05-24 05:20:50 [__init__.py:30] Available plugins for group vllm.general_plugins:
INFO 05-24 05:20:50 [__init__.py:32] name=lora_filesystem_resolver, value=vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-24 05:20:50 [__init__.py:34] all available plugins for group vllm.general_plugins will be loaded.
INFO 05-24 05:20:50 [__init__.py:36] set environment variable VLLM_PLUGINS to control which plugins to load.
INFO 05-24 05:20:50 [__init__.py:44] plugin lora_filesystem_resolver loaded.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 05-24 05:21:05 [config.py:788] This model supports multiple tasks: {'embed', 'generate', 'score', 'reward', 'classify'}. Defaulting to 'generate'.
INFO 05-24 05:21:05 [config.py:2115] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

WARNING 05-24 05:21:07 [utils.py:2530] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized


In [ ]:
sampling_params = SamplingParams(temperature=0, max_tokens=100)

# Querying the age of John Doe
get_generation_time(
    llm,
    sampling_params,
    LONG_PROMPT +
    "Question: what is the age of John Doe? Your answer: The age of John Doe is ",
)

# Querying the age of Zack Blue
# This query will be faster since vllm avoids computing the KV cache of LONG_PROMPT again.
get_generation_time(
    llm,
    sampling_params,
    LONG_PROMPT +
    "Question: what is the age of Zack Blue? Your answer: The age of Zack Blue is ",
)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

------------------------------
Output:                                                                                                     
Generation time: 0.271160364151001 seconds.
------------------------------


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

------------------------------
Output:                                                                                                     
Generation time: 0.26052212715148926 seconds.
------------------------------
